# 1. Initializations

## 1.1 General imports

In [ ]:
### global
import logging
from smartcheck.logger_config import setup_logger
setup_logger(logging.INFO)
from itertools import islice

### data management
import pandas as pd
import numpy as np

### machine learning (scikit-learn)
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import shap

### graphical
import matplotlib.pyplot as plt
# for jupyter notebook management
%matplotlib inline
import seaborn as sns


## 1.2 General dataframe functions

In [ ]:
import smartcheck.dataframe_common as dfc
import smartcheck.dataframe_project_specific as dfps

## 1.3 Specific preprocessing classes

In [ ]:
import smartcheck.preprocessing_project_specific as pps

# 2. Loading and Preprocessing

In [ ]:
df_cpt_raw = dfc.load_dataset_from_config('velo_comptage_ml_ready_data', sep=',', index_col=0)

if df_cpt_raw is not None and isinstance(df_cpt_raw, pd.DataFrame):
    df_cpt = df_cpt_raw.copy()

In [ ]:
df_cpt.info()

## 2.1 Preprocessing pipelines

In [ ]:
keep_cols = [
    "nom_du_site_de_comptage",
    "comptage_horaire",
    "date_et_heure_de_comptage",
    "orientation_compteur",
    "latitude",
    "longitude",
    "arrondissement",
    "jour_ferie",
    "vacances_scolaires",
    "temperature_2m_c",
    "rain_mm",
    "snowfall_cm",
    # "weather_code_wmo_code",
    "elevation",
    "weather_code_wmo_code_category",
]

pipe_preproc = Pipeline([
    ("filter_columns", pps.ColumnFilterTransformer(columns_to_keep=keep_cols)),
    ("add_datetime_features", pps.DatetimePeriodicsTransformer(timestamp_col="date_et_heure_de_comptage")),
])

df_preproc = pipe_preproc.fit_transform(df_cpt)
if df_preproc is not None and isinstance(df_preproc, pd.DataFrame):
    df = df_preproc.copy()

In [ ]:
# Verification des distributions après preprocessing
df.info()
display(df.select_dtypes(include=np.number).describe())
display(df.select_dtypes(include='object').describe())

#### Iterative manual feature selections with VIF

In [ ]:
# VarianceInflationFactor (VIF)
# - VIF > 5 → multicolinéarité forte
# - VIF > 10 → à supprimer quasi sûr
X = df.drop(columns='comptage_horaire').select_dtypes(include=np.number)
X = pd.DataFrame(X.astype("float64"))
vif_data = pd.DataFrame()
vif_data["feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(vif_data)

# Matrice de correlation 
plt.figure(figsize=(20, 20))  # Taille de la figure
corr_mat = X.corr()
sns.heatmap(corr_mat, annot=True, fmt=".2f", cmap="coolwarm", mask=np.triu(corr_mat), center=0)
plt.show()

>🔥 Corrélations proches de ±1 :
>
>| Couple de variables                                   | Corrélation | Action recommandée              |
>| ----------------------------------------------------- | ----------- | ------------------------------- |
>| `day_of_year` ↔ `day`                                 | **1.00**    | ❌ garder un seul des deux       |
>| `month` ↔ `sin_month` / `cos_month`                   | ±0.67/0.56  | ❌ garder `sin/cos`, pas `month` |
>| `week` ↔ `sin_week` / `cos_week`                      | ±0.97/0.93  | ❌ garder `sin/cos`, pas `week`  |
>| `hour` ↔ `sin_hour`                                   | **–0.78**   | ❌ idem, `sin/cos_hour` > `hour` |
>| `day_of_week` ↔ `sin_day_of_week` / `cos_day_of_week` | \~±0.75     | ❌ idem, `sin/cos` > brut        |


In [ ]:
# Ajustement itération 1
col_to_drop1 = [
    'date_et_heure_de_comptage_day_of_year',
    'date_et_heure_de_comptage_year',
    'date_et_heure_de_comptage_month',
    'date_et_heure_de_comptage_day',
    'date_et_heure_de_comptage_day_of_week',
    'date_et_heure_de_comptage_hour'
]
X1 = X.drop(columns=col_to_drop1)

In [ ]:
# VarianceInflationFactor (VIF)
# - VIF > 5 → multicolinéarité forte
# - VIF > 10 → à supprimer quasi sûr
vif_data = pd.DataFrame()
vif_data["feature"] = X1.columns
vif_data["VIF"] = [variance_inflation_factor(X1.values, i) for i in range(X1.shape[1])]
print(vif_data)

# Matrice de correlation 
plt.figure(figsize=(20, 20))  # Taille de la figure
corr_mat = X1.corr()
sns.heatmap(corr_mat, annot=True, fmt=".2f", cmap="coolwarm", mask=np.triu(corr_mat), center=0)
plt.show()

In [ ]:
# Ajustement itération 2
col_to_drop2 = [
    'latitude',
    'longitude',
    'date_et_heure_de_comptage_sin_month',
    'date_et_heure_de_comptage_sin_week',
]
X2 = X1.drop(columns=col_to_drop2)

In [ ]:
# VarianceInflationFactor (VIF)
# - VIF > 5 → multicolinéarité forte
# - VIF > 10 → à supprimer quasi sûr
vif_data = pd.DataFrame()
vif_data["feature"] = X2.columns
vif_data["VIF"] = [variance_inflation_factor(X2.values, i) for i in range(X2.shape[1])]
print(vif_data)

# Matrice de correlation 
plt.figure(figsize=(20, 20))  # Taille de la figure
corr_mat = X2.corr()
sns.heatmap(corr_mat, annot=True, fmt=".2f", cmap="coolwarm", mask=np.triu(corr_mat), center=0)
plt.show()

In [ ]:
# Ajustement itération 3
col_to_drop3 = [
    'date_et_heure_de_comptage_cos_month',
    'date_et_heure_de_comptage_week',
]
X3 = X2.drop(columns=col_to_drop3)

In [ ]:
# VarianceInflationFactor (VIF)
# - VIF > 5 → multicolinéarité forte
# - VIF > 10 → à supprimer quasi sûr
vif_data = pd.DataFrame()
vif_data["feature"] = X3.columns
vif_data["VIF"] = [variance_inflation_factor(X3.values, i) for i in range(X3.shape[1])]
print(vif_data)

# Matrice de correlation 
plt.figure(figsize=(20, 20))  # Taille de la figure
corr_mat = X3.corr()
sns.heatmap(corr_mat, annot=True, fmt=".2f", cmap="coolwarm", mask=np.triu(corr_mat), center=0)
plt.show()

In [ ]:
df = df.drop(columns=col_to_drop1+col_to_drop2+col_to_drop3)
df_feat_sel = df.copy()

## 2.2 Column Transformers

#### Categorical and Numerical

In [ ]:
num_col = list(df.drop(columns='comptage_horaire').select_dtypes(include=np.number).columns)
cat_col = list(df.select_dtypes(include='object').columns)
# s_scaler = StandardScaler()
mm_scaler = MinMaxScaler()
ohe_enc = OneHotEncoder(
    # drop='first', # évites la multicolinéarité mais déclenche des warning (les catégories inconnues participent a renforcer la catégorie droppée...)
    handle_unknown='ignore'
)
tr_num_col = Pipeline(
    steps = [
        ('standardisation', mm_scaler)
    ]
)
tr_cat_col = Pipeline(
    steps = [
        ('encoder', ohe_enc)
    ]
)
tr_columns = ColumnTransformer(
    transformers=[ 
        ('num_col_transf', tr_num_col, num_col),
        ('cat_col_transf', tr_cat_col, cat_col)
    ]
)

#### Test de la pipeline column transformer

In [ ]:
pipe_columns = Pipeline(
    steps= [
        ('preprocessing_column_transformation', tr_columns), 
    ]
)

In [ ]:
# Fit la pipeline
array_tr = pipe_columns.fit_transform(df)
# Si sparse matrix, convertir en dense
if hasattr(array_tr, "toarray"):
    array_tr = array_tr.toarray()  # type: ignore
# Récupération des noms de colonnes
num_col_names = num_col  # inchangés
# Récupérer l'encodeur entraîné depuis la pipeline
ohe_encoder_fitted = (
    pipe_columns.named_steps['preprocessing_column_transformation']
    .named_transformers_['cat_col_transf']  
)
# Récupérer les noms de colonnes encodées
cat_col_names = ohe_encoder_fitted.get_feature_names_out(cat_col)
# Concaténation des noms
all_feature_names = np.concatenate([num_col_names, cat_col_names])
# Création du DataFrame
df_tr = pd.DataFrame(array_tr, columns=all_feature_names, index=df.index)

In [ ]:
df_tr.describe()

# 3. Regression modeling

## 3.0 Common functions

In [ ]:
# Visualisation sur periode a partir des resultat du modele
def afficher_resultats_modele_temporel(compteur, model_results, periode_limite=('2025-04-01', '2025-04-16')):
    # pipe_generic = model_results[0]
    # X_train = model_results[1]
    # X_train_dates = model_results[2]
    # X_test = model_results[3]
    X_test_dates = model_results[4]
    # y_train = model_results[5]
    y_test = model_results[6]
    y_test_pred = model_results[7]

    # Métriques
    r2 = r2_score(y_test, y_test_pred)
    rmse = root_mean_squared_error(y_test, y_test_pred)
    mae = mean_absolute_error(y_test, y_test_pred)

    # Affichage des métriques
    print(f"\n📊 Résultats pour le compteur {compteur}")
    print(f"  - R²     : {r2:.4f}")
    print(f"  - RMSE   : {rmse:.2f}")
    print(f"  - MAE    : {mae:.2f}")
    print("-" * 40)

    # Tracé des courbes de prédiction <=> valeur réelles
    plt.figure(figsize=(12, 8))
    plt.plot(X_test_dates.date_et_heure_de_comptage_local, y_test, label='Valeurs réelles')
    plt.plot(X_test_dates.date_et_heure_de_comptage_local, y_test_pred, label='Prédictions', linestyle='--')
    plt.title(f'Prédictions – Compteur {compteur}')
    plt.xlabel('Date')
    plt.xlim([pd.to_datetime(periode_limite[0]), pd.to_datetime(periode_limite[1])])
    plt.ylabel('Comptage Horaire')
    plt.legend()
    plt.grid(True)
    plt.show()

# Interprétabilité du modèle
def get_feature_names_from_column_transformer(ct):
    feature_names = []
    for name, transformer, cols in ct.transformers_:
        if name != 'remainder':
            if hasattr(transformer, 'get_feature_names_out'):
                trans_features = transformer.get_feature_names_out(cols)
            else:
                trans_features = cols
            feature_names.extend(trans_features)
        elif name == 'remainder' and transformer == 'passthrough':
            feature_names.extend(cols)
    return feature_names

def interpreter_resultats_modele_temporel(compteur, model_results, explain_with_shape=False):
    pipe_generic = model_results[0]
    # X_train = model_results[1]
    # X_train_dates = model_results[2]
    X_test = model_results[3]
    # X_test_dates = model_results[4]
    # y_train = model_results[5]
    # y_test = model_results[6]
    y_test_pred = model_results[7]

    # Parcours des étapes de la pipeline pour trouver le modèle associé et s'il existe de l'interprétabilité exploitable
    is_linear_model = None
    is_knn_model = None
    model_step_name = None
    for step_name, step in pipe_generic.named_steps.items():
        if isinstance(step, LinearRegression):
            model_step_name = step_name
            is_linear_model = True
            break
        if isinstance(step, KNeighborsRegressor):
            model_step_name = step_name
            is_knn_model = True
            break

    # Interpretation avec Shap :
    if explain_with_shape:
        explainer = shap.KernelExplainer(
            pipe_generic.named_steps[model_step_name].predict,
            data=pipe_generic.named_steps['preprocessing_column_transformation'].transform(X_test),
        )
        shap_values = explainer.shap_values(X_test)
        shap.summary_plot(shap_values, X_test, plot_type="bar", max_display=10)
        shap.summary_plot(shap_values, X_test, max_display=10)
        return


    # Interprétabilité pour les modèles de regression linéaire (visualisation de l'influence des variables via leur coefficients)
    if is_linear_model:
        feature_names = get_feature_names_from_column_transformer(
            pipe_generic.named_steps['preprocessing_column_transformation']
        )
        coeffs = pipe_generic.named_steps[model_step_name].coef_
        
        feat_importance = pd.Series(coeffs, index=feature_names)
        feat_importance.sort_values(ascending=False).plot(kind='barh', figsize=(15,20))
        plt.show()
    
    # Interprétabilité pour les modèles KNN (projection PCA des variables explicatives utilisées et colorées par les prédictions faites par KNN)
    if is_knn_model:
        # Récupération de X transformé (comme vu par le modèle)
        nb_feats = X_test.shape[1]
        fe_proj = PCA(n_components=2)
        X_test_transformed = fe_proj.fit_transform(
            pipe_generic.named_steps['preprocessing_column_transformation'].transform(X_test)
        )
        print("Variance expliquée par les composantes primaires PC1 et PC2 :", fe_proj.explained_variance_ratio_)
        if fe_proj.explained_variance_ratio_.sum() < 0.9:
            print(f"Variance totale expliquée [{fe_proj.explained_variance_ratio_.sum()}] insuffisante pour afficher un graphique pertinent (>=0.9 requis)")
            return
        coeff = fe_proj.components_.transpose()
        xs = X_test_transformed[:, 0]
        ys = X_test_transformed[:, 1]
        scalex = 1.0/(xs.max() - xs.min())
        scaley = 1.0/(ys.max() - ys.min())
        df_comp_proj = pd.DataFrame({'PC1': xs*scalex, 'PC2': ys * scaley})
        df_proj_final = pd.concat([
            df_comp_proj, 
            pd.Series(y_test_pred, name='comptage horaire prédit')], axis=1)

        # Affichage
        plt.figure(figsize=(12, 8))
        sns.scatterplot(x='PC1', y='PC2', hue='comptage horaire prédit', palette='coolwarm', data=df_proj_final, alpha=0.9)
        for i in range(nb_feats):
            plt.arrow(0, 0, coeff[i, 0]*1.5, coeff[i, 1]*1.5,
                      color='k', alpha=0.5, head_width=0.01)
            plt.text(coeff[i, 0]*1.5, coeff[i, 1]*1.5, 
                     X_test.columns[i], color='k')
        plt.title(f"Projection PCA des données pour compteur {compteur}")
        plt.xlim(-0.8, 0.8)
        plt.ylim(-0.8, 0.8)
        plt.show()

## 3.1 Linear Regresion

In [ ]:
pipe_linear_regression = Pipeline(
    steps= [
        ('preprocessing_column_transformation', tr_columns), 
        ('linear_regression_model',LinearRegression())
    ]
)

#### SANS AR(1) MA(24)

In [ ]:
models_lr_simple_results = {}
# pas d'aggrégation, juste un regroupement par nom de site et orientation (utilisé ensuite dans la boucle for)
grouped = df.groupby(["nom_du_site_de_comptage", "orientation_compteur"])
for compteur_id, df_compteur in grouped:
    # tri chrono + pipeline prétraitement + split
    df_compteur = df_compteur.sort_values("date_et_heure_de_comptage_local")
    X_train, X_train_dates, X_test, X_test_dates, y_train, y_test = \
        dfps.train_test_split_time_aware(
            df_compteur,
            timestamp_cols=["date_et_heure_de_comptage_utc", "date_et_heure_de_comptage_local"],
            target_col="comptage_horaire"
        )
    # pipeline + fit
    pipe_generic = pipe_linear_regression.fit(X_train, y_train)
    # ohe = preprocessor.named_transformers_['cat']
    # print("Colonnes encodées :", preprocessor.transformers_[0][2])
    y_test_pred = pipe_generic.predict(X_test)
    models_lr_simple_results[compteur_id] = [
        pipe_generic, 
        X_train, 
        X_train_dates, 
        X_test, 
        X_test_dates, 
        y_train, 
        y_test, 
        y_test_pred
    ]


for key in models_lr_simple_results:
    print(f"- {key}")

#### AVEC AR(1) MA(24)

In [ ]:
models_lr_ar1ma24_results = {}

# Grouping by site + orientation
grouped = df.groupby(["nom_du_site_de_comptage", "orientation_compteur"])

for compteur_id, df_compteur in grouped:
    df_compteur = df_compteur.sort_values("date_et_heure_de_comptage_local")

    # Chronological split
    X_train, X_train_dates, X_test, X_test_dates, y_train, y_test = dfps.train_test_split_time_aware(
        df_compteur,
        timestamp_cols=[
            "date_et_heure_de_comptage_utc",
            "date_et_heure_de_comptage_local"
        ],
        target_col="comptage_horaire"
    )
    # Fit AR feature transformer on training set
    tr_ar_feats = pps.AutoregressiveFeaturesTransformer(rolling_window=24)
    X_train_ar, X_train_dates_ar, y_train_ar = tr_ar_feats.fit_transform(
        X_train, X_train_dates, y_train
    )
    # Train model
    pipe_generic = pipe_linear_regression.fit(X_train_ar, y_train_ar)

    # Transform test set without refitting
    X_test_ar, X_test_dates_ar, y_test_ar = tr_ar_feats.transform_test(
        X_test, X_test_dates, y_test
    )

    # Predict
    y_test_pred = pipe_generic.predict(X_test_ar)

    # Save results
    models_lr_ar1ma24_results[compteur_id] = [
        pipe_generic,
        X_train_ar,
        X_train_dates_ar,
        X_test_ar,
        X_test_dates_ar,
        y_train_ar,
        y_test_ar,
        y_test_pred
    ]

for key in models_lr_ar1ma24_results:
    print(f"- {key}")

## 3.1 KNN Regresion

In [ ]:
pipe_knn_regression = Pipeline(
    steps= [
        ('preprocessing_column_transformation', tr_columns), 
        ('knn_regression_model',KNeighborsRegressor(n_jobs=-1, metric='minkowski'))
    ]
)

#### SANS AR(1) MA(24)

In [ ]:
models_knn_simple_results = {}
# pas d'aggrégation, juste un regroupement par nom de site et orientation (utilisé ensuite dans la boucle for)
grouped = df.groupby(["nom_du_site_de_comptage", "orientation_compteur"])
for compteur_id, df_compteur in grouped:
    # tri chrono + pipeline prétraitement + split
    df_compteur = df_compteur.sort_values("date_et_heure_de_comptage_local")
    X_train, X_train_dates, X_test, X_test_dates, y_train, y_test = \
        dfps.train_test_split_time_aware(
            df_compteur,
            timestamp_cols=["date_et_heure_de_comptage_utc", "date_et_heure_de_comptage_local"],
            target_col="comptage_horaire"
        )
    # pipeline + fit
    pipe_generic = pipe_knn_regression.fit(X_train, y_train)
    y_test_pred = pipe_generic.predict(X_test)
    models_knn_simple_results[compteur_id] = [
        pipe_generic, 
        X_train, 
        X_train_dates, 
        X_test, 
        X_test_dates, 
        y_train, 
        y_test, 
        y_test_pred
    ]

for key in models_knn_simple_results:
    print(f"- {key}")

#### AVEC AR(1) MA(24)

In [ ]:
models_knn_ar1ma24_results = {}

# Grouping by site + orientation
grouped = df.groupby(["nom_du_site_de_comptage", "orientation_compteur"])

for compteur_id, df_compteur in grouped:
    df_compteur = df_compteur.sort_values("date_et_heure_de_comptage_local")

    # Chronological split
    X_train, X_train_dates, X_test, X_test_dates, y_train, y_test = dfps.train_test_split_time_aware(
        df_compteur,
        timestamp_cols=[
            "date_et_heure_de_comptage_utc",
            "date_et_heure_de_comptage_local"
        ],
        target_col="comptage_horaire"
    )
    # Fit AR feature transformer on training set
    tr_ar_feats = pps.AutoregressiveFeaturesTransformer(rolling_window=24)
    X_train_ar, X_train_dates_ar, y_train_ar = tr_ar_feats.fit_transform(
        X_train, X_train_dates, y_train
    )
    # Train model
    pipe_generic = pipe_knn_regression.fit(X_train_ar, y_train_ar)

    # Transform test set without refitting
    X_test_ar, X_test_dates_ar, y_test_ar = tr_ar_feats.transform_test(
        X_test, X_test_dates, y_test
    )

    # Predict
    y_test_pred = pipe_generic.predict(X_test_ar)

    # Save results
    models_knn_ar1ma24_results[compteur_id] = [
        pipe_generic,
        X_train_ar,
        X_train_dates_ar,
        X_test_ar,
        X_test_dates_ar,
        y_train_ar,
        y_test_ar,
        y_test_pred
    ]

for key in models_knn_ar1ma24_results:
    print(f"- {key}")

# 4 Performance analysis and visualisation

#### compteur ('Totem 73 boulevard de Sébastopol', 'S-N')

In [ ]:
# Afficher une modélisation de compteur spécifique
compteur = ('Totem 73 boulevard de Sébastopol', 'S-N')
print("\n", "*"*80, "\n", "Modèle Régression linéaire simple :")
afficher_resultats_modele_temporel(compteur, models_lr_simple_results[compteur])
interpreter_resultats_modele_temporel(compteur, models_lr_simple_results[compteur])
print("\n", "*"*80, "\n", "Modèle Régression linéaire avec AR(1) MA(1) :")
afficher_resultats_modele_temporel(compteur, models_lr_ar1ma24_results[compteur])
interpreter_resultats_modele_temporel(compteur, models_lr_simple_results[compteur])
print("\n", "*"*80, "\n", "Modèle KNN simple :")
afficher_resultats_modele_temporel(compteur, models_knn_simple_results[compteur])
interpreter_resultats_modele_temporel(compteur, models_knn_simple_results[compteur])
print("\n", "*"*80, "\n", "Modèle KNN avec AR(1) MA(1) :")
afficher_resultats_modele_temporel(compteur, models_knn_ar1ma24_results[compteur])
interpreter_resultats_modele_temporel(compteur, models_knn_ar1ma24_results[compteur])

In [ ]:
# Limiter l'affichage/analyse à un sous-ensemble des X premiers compteurs de chaque résultats
nb_compteurs = 1
for compteur, model_results in islice(models_lr_simple_results.items(), nb_compteurs):
    afficher_resultats_modele_temporel(compteur, model_results)
for compteur, model_results in islice(models_lr_ar1ma24_results.items(), nb_compteurs):
    afficher_resultats_modele_temporel(compteur, model_results)
for compteur, model_results in islice(models_knn_simple_results.items(), nb_compteurs):
    afficher_resultats_modele_temporel(compteur, model_results)
for compteur, model_results in islice(models_knn_ar1ma24_results.items(), nb_compteurs):
    afficher_resultats_modele_temporel(compteur, model_results)

In [ ]:
# NB : le compteur 106 avenue Denfert Rochereau est bien à 0 tout le temps
display(df[df.nom_du_site_de_comptage=='106 avenue Denfert Rochereau'].comptage_horaire.sum())  # type: ignore